In [1]:
from deeprobust.graph.data import Dpr2Pyg, Pyg2Dpr
from deeprobust.graph.data import Dataset as DRDataset
import torch
from torch_geometric.data import Data
import numpy as np
from deeprobust.graph.data import Dataset, PrePtbDataset, PtbDataset
from deeprobust.graph.defense import GCN, RGCN, ProGNN, SimPGCN, GCNSVD, GCNJaccard, GAT
from deeprobust.graph.global_attack import Metattack, DICE, Random, PGDAttack
from scipy.sparse import csr_matrix
import torch
from torch.nn import Linear
import torch.nn.functional as F
from torch_geometric.transforms import Compose
from torch_geometric.datasets import Amazon
from torch_geometric.transforms.random_node_split import RandomNodeSplit
from torch_geometric.datasets import Planetoid
from torch_geometric.transforms import NormalizeFeatures
from torch_geometric.nn import GCNConv
from torch_geometric.nn import GATConv
from torch_geometric.loader import NeighborLoader
from torch_geometric.nn import SAGEConv
from sklearn.metrics import roc_auc_score

from torch_geometric.utils import negative_sampling
from torch_geometric.utils import train_test_split_edges

from copy import deepcopy
import torch.nn as nn
from IPython.display import Javascript  # Restrict height of output cell.
import matplotlib.pyplot as plt

In [2]:
seed = 15
# ptb_rates = [0.05, 0.1, 0.15, 0.2, 0.25, 0.3]
ptb_rates = [0.05, 0.1]
# ptb_rate = 0.25
# ptb_rate1 = 0.15
dataset = 'cora'
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)

In [3]:
data = DRDataset(root='/tmp/', name=dataset, setting='prognn')
adj, features, labels = data.adj, data.features, data.labels
idx_train, idx_val, idx_test = data.idx_train, data.idx_val, data.idx_test
idx_unlabeled = np.union1d(idx_val, idx_test)
idx_unlabeled = np.union1d(idx_val, idx_test)


device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
# budget = int(ptb_rate * (adj.todense().sum() // 2))
# print(budget)
# budget1 = int(ptb_rate1 * (adj.todense().sum() // 2))
# print(budget1)

Loading cora dataset...
Selecting 1 largest connected components


In [4]:
np.save('/tmp/cora_adj.npy',adj.todense())

In [5]:
np.save('/tmp/cora_features.npy',features.todense())

In [6]:
np.save('/tmp/cora_labels.npy',labels)

In [7]:
np.save('/tmp/cora_idx_test',idx_test)

# Metattack

## GSAGE

In [8]:
import sys
sys.path.append('../')
from util import *

In [9]:
new_data = Dpr2Pyg(DRDataset(root='/tmp/', name='cora'))[0]
surrogate = get_model('../', new_data.num_node_features, 64, 7, 'cora', Models.GSAGE)

Loading cora dataset...
Selecting 1 largest connected components


Processing...
/home/wcorcoran/anaconda3/envs/mesp/lib/python3.8/site-packages/deeprobust/graph/data/pyg_dataset.py:48: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at ../torch/csrc/utils/tensor_new.cpp:245.)
  edge_index = torch.LongTensor(dpr_data.adj.nonzero())
Done!
/home/wcorcoran/anaconda3/envs/mesp/lib/python3.8/site-packages/torch_geometric/data/in_memory_dataset.py:300: UserWarning: It is not recommended to directly access the internal storage format `data` of an 'InMemoryDataset'. If you are absolutely certain what you are doing, access the internal storage via `InMemoryDataset._data` instead to suppress this warning. Alternatively, you can access stacked individual attributes of every graph via `dataset.{attr_name}`.
  warnings.warn(msg)


In [10]:
benchmark_clean = test_model(surrogate, new_data, Models.GSAGE, testMask=True)

In [11]:
from copy import deepcopy

gsage_results = []

for ptb in ptb_rates:
  torch.cuda.empty_cache()
  budget = int(ptb * (adj.todense().sum() // 2))
  # Setup Attack Model
  model = Metattack(surrogate, nnodes=adj.shape[0], feature_shape=features.shape,
          attack_structure=True, attack_features=False, device=device, lambda_=0).to(device)
  # Attack
  model.attack(features, adj, labels, idx_train, idx_unlabeled, n_perturbations=budget, ll_constraint=False)
  modified_adj = model.modified_adj # modified_adj is a torch.tensor
  modified_adj = modified_adj.cpu().numpy()
  modified_adj = csr_matrix(modified_adj)

  perturbed_adj = deepcopy(modified_adj)
  # atk_model = GCN(nfeat=features.shape[1], nclass=labels.max().item()+1,
  #               nhid=16, dropout=0, with_relu=True, with_bias=True, device=device).to(device)
  # atk_model.fit(features, modified_adj, labels, idx_train, idx_val, patience=100, verbose=True)

  # atk_acc = atk_model.test(idx_test)
  # print("accuracy: ", atk_acc)
  # print("benchmark change: ", atk_acc - benchmark_clean)
  # gsage_results.append(atk_acc - benchmark_clean)


AttributeError: 'NoneType' object has no attribute 'argmax'

## GCN

In [8]:
# Setup Surrogate model
surrogate = GCN(nfeat=features.shape[1], nclass=labels.max().item()+1,
                nhid=16, dropout=0, with_relu=True, with_bias=True, device=device).to(device)
surrogate.fit(features, adj, labels, idx_train, idx_val, patience=100, verbose=True)

=== training gcn model ===
Epoch 0, training loss: 1.9264339208602905
Epoch 10, training loss: 0.8777099251747131
Epoch 20, training loss: 0.263236403465271
Epoch 30, training loss: 0.0835253894329071
Epoch 40, training loss: 0.03957008570432663
Epoch 50, training loss: 0.029189568012952805
Epoch 60, training loss: 0.028210503980517387
Epoch 70, training loss: 0.02904457040131092
Epoch 80, training loss: 0.02885228581726551
Epoch 90, training loss: 0.027361346408724785
Epoch 100, training loss: 0.02537829615175724
Epoch 110, training loss: 0.023561256006360054
Epoch 120, training loss: 0.02213549241423607
Epoch 130, training loss: 0.020990343764424324
=== early stopping at 131, loss_val = 0.5125710964202881 ===


In [9]:
preds=surrogate.predict(features,adj)
_, predicted_classes = torch.max(preds, 1)  # Returns values, indices; we need indices

# Step 3: Subset for test nodes
test_preds = predicted_classes[idx_test].cpu().numpy()
test_labels = labels[idx_test]

# Step 4: Calculate accuracy
correct = (test_preds == test_labels).sum().item()  # Count correct predictions
total = idx_test.shape[0]  # Total number of test nodes
test_accuracy = correct / total

print(f'Test Accuracy: {test_accuracy}')

Test Accuracy: 0.8329979879275654


In [10]:
benchmark_clean = test_accuracy

In [11]:
from copy import deepcopy

gcn_results = []

for ptb in ptb_rates:
  torch.cuda.empty_cache()
  budget = int(ptb * (adj.todense().sum() // 2))
  # Setup Attack Model
  model = Metattack(surrogate, nnodes=adj.shape[0], feature_shape=features.shape,
          attack_structure=True, attack_features=False, device=device, lambda_=0).to(device)
  # Attack
  model.attack(features, adj, labels, idx_train, idx_unlabeled, n_perturbations=budget, ll_constraint=False)
  modified_adj = model.modified_adj # modified_adj is a torch.tensor
  modified_adj = modified_adj.cpu().numpy()
  modified_adj = csr_matrix(modified_adj)

  perturbed_adj = deepcopy(modified_adj)
  atk_model = GCN(nfeat=features.shape[1], nclass=labels.max().item()+1,
                nhid=16, dropout=0, with_relu=True, with_bias=True, device=device).to(device)
  atk_model.fit(features, modified_adj, labels, idx_train, idx_val, patience=100, verbose=True)

  atk_acc = atk_model.test(idx_test)
  print("accuracy: ", atk_acc)
  print("benchmark change: ", atk_acc - benchmark_clean)
  gcn_results.append(atk_acc - benchmark_clean)


Perturbing graph:   0%|          | 0/253 [00:00<?, ?it/s]

GCN loss on unlabled data: 0.5205798745155334
GCN acc on unlabled data: 0.8359409924005364
attack loss: 0.3264505863189697


Perturbing graph:   0%|          | 1/253 [00:00<01:51,  2.27it/s]

GCN loss on unlabled data: 0.5136726498603821
GCN acc on unlabled data: 0.835493965131873
attack loss: 0.333562970161438


Perturbing graph:   1%|          | 2/253 [00:00<01:48,  2.30it/s]

...

GCN loss on unlabled data: 0.8552752733230591
GCN acc on unlabled data: 0.7684398748323648
attack loss: 0.7424120903015137


Perturbing graph:  45%|████▍     | 113/253 [00:48<01:01,  2.27it/s]

GCN loss on unlabled data: 0.8329433798789978
GCN acc on unlabled data: 0.75592311130979
attack loss: 0.7162291407585144


Perturbing graph:  45%|████▌     | 114/253 [00:49<00:59,  2.33it/s]


KeyboardInterrupt: 

In [ ]:
gcn_results

## GCNJaccard

In [ ]:
surrogate1 = GCNJaccard(nfeat=features.shape[1],
          nhid=16,
          nclass=labels.max().item() + 1,
          dropout=0.5, device=device).to(device)
surrogate1.fit(features, adj, labels, idx_train, idx_val, threshold=0.03)

In [ ]:
preds=surrogate1.predict(features,adj)
_, predicted_classes = torch.max(preds, 1)  # Returns values, indices; we need indices

# Step 3: Subset for test nodes
test_preds = predicted_classes[idx_test].cpu().numpy()
test_labels = labels[idx_test]

# Step 4: Calculate accuracy
correct = (test_preds == test_labels).sum().item()  # Count correct predictions
total = idx_test.shape[0]  # Total number of test nodes
test_accuracy = correct / total
benchmark_clean = test_accuracy

print(f'Test Accuracy: {test_accuracy}')

In [ ]:
from copy import deepcopy

ptb_rates = [0.1, 0.2, 0.3, 0.4, 0.5]
jaccard_results = []

for ptb in ptb_rates:
  budget = int(ptb * (adj.todense().sum() // 2))
  # Setup Attack Model
  model = Metattack(surrogate1, nnodes=adj.shape[0], feature_shape=features.shape,
          attack_structure=True, attack_features=False, device=device, lambda_=0).to(device)
  # Attack
  model.attack(features, adj, labels, idx_train, idx_unlabeled, n_perturbations=budget, ll_constraint=False)
  modified_adj = model.modified_adj # modified_adj is a torch.tensor
  modified_adj = modified_adj.cpu().numpy()
  modified_adj = csr_matrix(modified_adj)

  perturbed_adj = deepcopy(modified_adj)
  atk_model = GCNJaccard(nfeat=features.shape[1],
            nhid=16,
            nclass=labels.max().item() + 1,
            dropout=0.5, device=device).to(device)
  atk_model.fit(features, modified_adj, labels, idx_train, idx_val, patience=100, verbose=True)

  atk_acc = atk_model.test(idx_test)

  print("accuracy: ", atk_acc)
  print("benchmark change: ", atk_acc - benchmark_clean)
  jaccard_results.append(atk_acc - benchmark_clean)


## GAT

In [ ]:
surrogate5 = GAT(nfeat=features.shape[1],
      nhid=8, heads=8,
      nclass=labels.max().item() + 1,
      dropout=0.5, device=device)
surrogate5 = surrogate5.to(device)

pyg_data = Dpr2Pyg(data)
surrogate5.fit(pyg_data, verbose=True) # train with earlystopping
surrogate5.test()

In [ ]:
surrogate5.eval()
preds=surrogate5.predict(features,adj)
_, predicted_classes = torch.max(preds, 1)  # Returns values, indices; we need indices

# Step 3: Subset for test nodes
test_preds = predicted_classes[idx_test].cpu().numpy()
test_labels = labels[idx_test]

# Step 4: Calculate accuracy
correct = (test_preds == test_labels).sum().item()  # Count correct predictions
total = idx_test.shape[0]  # Total number of test nodes
test_accuracy = correct / total
benchmark_clean = test_accuracy

print(f'Test Accuracy: {test_accuracy}')

In [ ]:
from copy import deepcopy

ptb_rates = [0.1, 0.2, 0.3, 0.4, 0.5]
gat_results = []

for ptb in ptb_rates:
  budget = int(ptb * (adj.todense().sum() // 2))
  # Setup Attack Model
  model = Metattack(surrogate5, nnodes=adj.shape[0], feature_shape=features.shape,
          attack_structure=True, attack_features=False, device=device, lambda_=0).to(device)
  # Attack
  model.attack(features, adj, labels, idx_train, idx_unlabeled, n_perturbations=budget, ll_constraint=False)
  modified_adj = model.modified_adj # modified_adj is a torch.tensor
  modified_adj = modified_adj.cpu().numpy()
  modified_adj = csr_matrix(modified_adj)

  perturbed_adj = deepcopy(modified_adj)
  atk_model = GAT(nfeat=features.shape[1],
        nhid=8, heads=8,
        nclass=labels.max().item() + 1,
        dropout=0.5, device=device)
  atk_model = surrogate5.to(device)
  atk_model.fit(features, modified_adj, labels, idx_train, idx_val, patience=100, verbose=True)

  atk_acc = atk_model.test(idx_test)

  print("accuracy: ", atk_acc)
  print("benchmark change: ", atk_acc - benchmark_clean)
  gat_results.append(atk_acc - benchmark_clean)

## GSAGE

In [ ]:
# from deeprobust.graph.defense_pyg import SAGE

# surrogate6 = SAGE(nfeat=features.shape[1],
#       nhid=64, nclass=labels.max().item() + 1,
#       num_layers=2, device=device)
# surrogate6 = surrogate6.to(device)

# pyg_data = Dpr2Pyg(data)
# surrogate6.fit(pyg_data, verbose=True) # train with earlystopping
# surrogate6.test()

## Plotting

In [ ]:
# Plotting
plt.plot(ptb_rates, gcn_results, label="GCN")
plt.plot(ptb_rates, jaccard_results, label="Jaccard")
plt.plot(ptb_rates, gat_results, label="GAT")
plt.plot(ptb_rates, gsage_results, label="GSAGE")
plt.plot(ptb_rates, gsaint_results, label="GSAINT")

# Adding labels and legend
plt.xlabel("ptb_rates")
plt.ylabel("Results")
plt.legend()

# Show plot
plt.show()

## GCN

In [ ]:
# Setup Surrogate model
surrogate = GCN(nfeat=features.shape[1], nclass=labels.max().item()+1,
                nhid=64, dropout=0, with_relu=True, with_bias=True, device=device).to(device)
surrogate.fit(features, adj, labels, idx_train, idx_val, patience=100, verbose=True)

=== training gcn model ===
Epoch 0, training loss: 2.0013506412506104
Epoch 10, training loss: 0.1784532070159912
Epoch 20, training loss: 0.02265828847885132
Epoch 30, training loss: 0.008995148353278637
Epoch 40, training loss: 0.01016619335860014
Epoch 50, training loss: 0.014811103232204914
Epoch 60, training loss: 0.01746569201350212
Epoch 70, training loss: 0.016285421326756477
Epoch 80, training loss: 0.01452041044831276
Epoch 90, training loss: 0.01350044459104538
Epoch 100, training loss: 0.012842738069593906
Epoch 110, training loss: 0.012275326065719128
=== early stopping at 113, loss_val = 0.4966380298137665 ===


In [ ]:
preds=surrogate.predict(features,adj)
_, predicted_classes = torch.max(preds, 1)  # Returns values, indices; we need indices

# Step 3: Subset for test nodes
test_preds = predicted_classes[idx_test].cpu().numpy()
test_labels = labels[idx_test]

# Step 4: Calculate accuracy
correct = (test_preds == test_labels).sum().item()  # Count correct predictions
total = idx_test.shape[0]  # Total number of test nodes
test_accuracy = correct / total

print(f'Test Accuracy: {test_accuracy}')

Test Accuracy: 0.8350100603621731


In [ ]:
# Setup Attack Model
model = DICE(surrogate, nnodes=adj.shape[0], attack_structure=True, attack_features=False, device=device).to(device)
model.attack(adj, labels, n_perturbations=budget)
modified_adj = model.modified_adj
# Convert SciPy sparse matrix to PyTorch tensor
modified_adj_torch = torch.tensor(modified_adj.toarray(), dtype=torch.float)

# Now you can safely call .cpu() if needed (useful if working with CUDA tensors)
modified_adj_torch = modified_adj_torch.cpu()

modified_adj = csr_matrix(modified_adj_torch)

number of pertubations: 253


In [ ]:
atk_model = GCN(nfeat=features.shape[1], nclass=labels.max().item()+1,
                nhid=64, dropout=0, with_relu=True, with_bias=True, device=device).to(device)
atk_model.fit(features, modified_adj, labels, idx_train, idx_val, patience=100, verbose=True)

atk_acc = atk_model.test(idx_test)
print(atk_acc)

=== training gcn model ===
Epoch 0, training loss: 1.9988664388656616
Epoch 10, training loss: 0.1770801693201065
Epoch 20, training loss: 0.02095688320696354
Epoch 30, training loss: 0.008767915889620781
Epoch 40, training loss: 0.010151972994208336
Epoch 50, training loss: 0.01509387232363224
Epoch 60, training loss: 0.017798857763409615
Epoch 70, training loss: 0.016668790951371193
Epoch 80, training loss: 0.014997268095612526
Epoch 90, training loss: 0.01406441256403923
Epoch 100, training loss: 0.013466321863234043
Epoch 110, training loss: 0.012926824390888214
=== early stopping at 112, loss_val = 0.5503204464912415 ===
Test set results: loss= 0.5443 accuracy= 0.8234
0.8234406438631792


In [ ]:
print((atk_acc - benchmark_clean)*100)

-2.4144869215291687


## GCNSVD

In [ ]:
surrogate2 = GCNSVD(nfeat=features.shape[1],
          nhid=64,
          nclass=labels.max().item() + 1,
          dropout=0.5, device=device).to(device)
surrogate2.fit(features, adj, labels, idx_train, idx_val, k=50)

=== GCN-SVD: rank=50 ===
rank_after = 50
=== training gcn model ===
Epoch 0, training loss: 1.9225667715072632
Epoch 10, training loss: 0.3862052857875824
Epoch 20, training loss: 0.20534750819206238
Epoch 30, training loss: 0.14677411317825317
Epoch 40, training loss: 0.11764820665121078
Epoch 50, training loss: 0.1066364124417305
Epoch 60, training loss: 0.0935211107134819
Epoch 70, training loss: 0.09862826019525528
Epoch 80, training loss: 0.08794967085123062
Epoch 90, training loss: 0.08464223146438599
Epoch 100, training loss: 0.07630077749490738
Epoch 110, training loss: 0.06864850968122482
Epoch 120, training loss: 0.07030574232339859
Epoch 130, training loss: 0.06677495688199997
Epoch 140, training loss: 0.06452471762895584
Epoch 150, training loss: 0.06983944028615952
Epoch 160, training loss: 0.06885287910699844
Epoch 170, training loss: 0.053785767406225204
Epoch 180, training loss: 0.06173109635710716
Epoch 190, training loss: 0.07087459415197372
=== picking the best model

In [ ]:
preds=surrogate2.predict(features,adj)
_, predicted_classes = torch.max(preds, 1)  # Returns values, indices; we need indices

# Step 3: Subset for test nodes
test_preds = predicted_classes[idx_test].cpu().numpy()
test_labels = labels[idx_test]

# Step 4: Calculate accuracy
correct = (test_preds == test_labels).sum().item()  # Count correct predictions
total = idx_test.shape[0]  # Total number of test nodes
test_accuracy = correct / total

print(f'Test Accuracy: {test_accuracy}')

=== GCN-SVD: rank=50 ===
rank_after = 50
Test Accuracy: 0.778672032193159


In [ ]:
# Setup Attack Model
model = DICE(surrogate2, nnodes=adj.shape[0], attack_structure=True, attack_features=False, device=device).to(device)
model.attack(adj, labels, n_perturbations=budget)

modified_adj = model.modified_adj
# Convert SciPy sparse matrix to PyTorch tensor
modified_adj_torch = torch.tensor(modified_adj.toarray(), dtype=torch.float)

# Now you can safely call .cpu() if needed (useful if working with CUDA tensors)
modified_adj_torch = modified_adj_torch.cpu()

modified_adj = csr_matrix(modified_adj_torch)

number of pertubations: 253


In [ ]:
atk_model = GCNSVD(nfeat=features.shape[1],
          nhid=64,
          nclass=labels.max().item() + 1,
          dropout=0.5, device=device).to(device)
atk_model.fit(features, modified_adj, labels, idx_train, idx_val, patience=100, verbose=True)

atk_acc = atk_model.test(idx_test)
print(atk_acc)

=== GCN-SVD: rank=50 ===
rank_after = 50
=== training gcn model ===
Epoch 0, training loss: 1.9886462688446045
Epoch 10, training loss: 0.4526948034763336
Epoch 20, training loss: 0.2588203251361847
Epoch 30, training loss: 0.17912545800209045
Epoch 40, training loss: 0.14748014509677887
Epoch 50, training loss: 0.14473453164100647
Epoch 60, training loss: 0.1214304119348526
Epoch 70, training loss: 0.11344599723815918
Epoch 80, training loss: 0.11132913082838058
Epoch 90, training loss: 0.09962953627109528
Epoch 100, training loss: 0.09191514551639557
Epoch 110, training loss: 0.08824841678142548
Epoch 120, training loss: 0.10258487612009048
Epoch 130, training loss: 0.07447312027215958
Epoch 140, training loss: 0.08269505202770233
Epoch 150, training loss: 0.08718709647655487
Epoch 160, training loss: 0.07425755262374878
Epoch 170, training loss: 0.0755540207028389
Epoch 180, training loss: 0.07198939472436905
Epoch 190, training loss: 0.07253514230251312
=== picking the best model a

In [ ]:
print((atk_acc - benchmark_clean)*100)

-8.55130784708249


## GCNJaccard

In [ ]:
surrogate1 = GCNJaccard(nfeat=features.shape[1],
          nhid=64,
          nclass=labels.max().item() + 1,
          dropout=0.5, device=device).to(device)
surrogate1.fit(features, adj, labels, idx_train, idx_val, threshold=0.03)

removed 1015 edges in the original graph
=== training gcn model ===
Epoch 0, training loss: 1.974871039390564
Epoch 10, training loss: 0.20419029891490936
Epoch 20, training loss: 0.03915862366557121
Epoch 30, training loss: 0.019494246691465378
Epoch 40, training loss: 0.015022391453385353
Epoch 50, training loss: 0.020247647538781166
Epoch 60, training loss: 0.018639886751770973
Epoch 70, training loss: 0.023240288719534874
Epoch 80, training loss: 0.019225651398301125
Epoch 90, training loss: 0.017367009073495865
Epoch 100, training loss: 0.012840601615607738
Epoch 110, training loss: 0.01236194558441639
Epoch 120, training loss: 0.017497288063168526
Epoch 130, training loss: 0.015027550049126148
Epoch 140, training loss: 0.016727082431316376
Epoch 150, training loss: 0.012636402621865273
Epoch 160, training loss: 0.016095105558633804
Epoch 170, training loss: 0.01243385300040245
Epoch 180, training loss: 0.011558675207197666
Epoch 190, training loss: 0.01197216659784317
=== picking

In [ ]:
preds=surrogate1.predict(features,adj)
_, predicted_classes = torch.max(preds, 1)  # Returns values, indices; we need indices

# Step 3: Subset for test nodes
test_preds = predicted_classes[idx_test].cpu().numpy()
test_labels = labels[idx_test]

# Step 4: Calculate accuracy
correct = (test_preds == test_labels).sum().item()  # Count correct predictions
total = idx_test.shape[0]  # Total number of test nodes
test_accuracy = correct / total

print(f'Test Accuracy: {test_accuracy}')

removed 1015 edges in the original graph
Test Accuracy: 0.8204225352112676


In [ ]:
# Setup Attack Model
model = DICE(surrogate1, nnodes=adj.shape[0], attack_structure=True, attack_features=False, device=device).to(device)
model.attack(adj, labels, n_perturbations=budget)

modified_adj = model.modified_adj
# Convert SciPy sparse matrix to PyTorch tensor
modified_adj_torch = torch.tensor(modified_adj.toarray(), dtype=torch.float)

# Now you can safely call .cpu() if needed (useful if working with CUDA tensors)
modified_adj_torch = modified_adj_torch.cpu()

modified_adj = csr_matrix(modified_adj_torch)

number of pertubations: 253


In [ ]:
atk_model = GCNJaccard(nfeat=features.shape[1],
          nhid=64,
          nclass=labels.max().item() + 1,
          dropout=0.5, device=device).to(device)
atk_model.fit(features, modified_adj, labels, idx_train, idx_val, patience=100, verbose=True)

atk_acc = atk_model.test(idx_test)
print(atk_acc)

removed 592 edges in the original graph
=== training gcn model ===
Epoch 0, training loss: 1.9543516635894775
Epoch 10, training loss: 0.23513753712177277
Epoch 20, training loss: 0.0553223118185997
Epoch 30, training loss: 0.017869023606181145
Epoch 40, training loss: 0.01658315770328045
Epoch 50, training loss: 0.02238362282514572
Epoch 60, training loss: 0.018894152715802193
Epoch 70, training loss: 0.02389240451157093
Epoch 80, training loss: 0.017526455223560333
Epoch 90, training loss: 0.020820094272494316
Epoch 100, training loss: 0.02099476009607315
Epoch 110, training loss: 0.016860488802194595
Epoch 120, training loss: 0.015659354627132416
Epoch 130, training loss: 0.0217744130641222
Epoch 140, training loss: 0.01514825876802206
Epoch 150, training loss: 0.016387227922677994
Epoch 160, training loss: 0.01557573489844799
Epoch 170, training loss: 0.015020075254142284
Epoch 180, training loss: 0.013103967532515526
Epoch 190, training loss: 0.01770988292992115
=== picking the be

In [ ]:
benchmark_clean

0.8475855130784709

In [ ]:
print((atk_acc - benchmark_clean)*100)

-3.9738430583501017


## SimPGCN

In [ ]:
surrogate3 = SimPGCN(nnodes=features.shape[0], nfeat=features.shape[1],
    nhid=16, nclass=labels.max()+1, device=device).to(device)

surrogate3.fit(features, adj, labels, idx_train, idx_val, train_iters=200, verbose=True)

/usr/local/lib/python3.10/dist-packages/deeprobust/graph/utils.py:356: UserWarning: torch.sparse.SparseTensor(indices, values, shape, *, device=) is deprecated.  Please use torch.sparse_coo_tensor(indices, values, shape, dtype=, device=). (Triggered internally at ../torch/csrc/utils/tensor_new.cpp:618.)
  return torch.sparse.FloatTensor(sparseconcat.t(),sparsedata,torch.Size(sparse_mx.shape))


=== training gcn model ===
loading saved_knn/cosine_sims_(2485, 1433).npy
number of sampled: 21972
Epoch 0, training loss: 1.9035120010375977
Epoch 10, training loss: 1.3902671337127686
Epoch 20, training loss: 0.9233468174934387
Epoch 30, training loss: 0.491983562707901
Epoch 40, training loss: 0.212081179022789
Epoch 50, training loss: 0.09256990998983383
Epoch 60, training loss: 0.048811428248882294
Epoch 70, training loss: 0.03520660102367401
Epoch 80, training loss: 0.022107426077127457
Epoch 90, training loss: 0.01542553212493658
Epoch 100, training loss: 0.015348459593951702
Epoch 110, training loss: 0.03367571532726288
Epoch 120, training loss: 0.02575494721531868
Epoch 130, training loss: 0.0166032537817955
Epoch 140, training loss: 0.02154587209224701
Epoch 150, training loss: 0.01488436572253704
Epoch 160, training loss: 0.011220656335353851
Epoch 170, training loss: 0.011381352320313454
Epoch 180, training loss: 0.01340511441230774
Epoch 190, training loss: 0.0183904431760

In [ ]:
preds=surrogate3.predict(features,adj)
_, predicted_classes = torch.max(preds, 1)  # Returns values, indices; we need indices

# Step 3: Subset for test nodes
test_preds = predicted_classes[idx_test].cpu().numpy()
test_labels = labels[idx_test]

# Step 4: Calculate accuracy
correct = (test_preds == test_labels).sum().item()  # Count correct predictions
total = idx_test.shape[0]  # Total number of test nodes
test_accuracy = correct / total

print(f'Test Accuracy: {test_accuracy}')

Test Accuracy: 0.8254527162977867


In [ ]:
# Setup Attack Model
model = DICE(surrogate3, nnodes=adj.shape[0], attack_structure=True, attack_features=False, device=device).to(device)
model.attack(adj, labels, n_perturbations=budget)

modified_adj = model.modified_adj
# Convert SciPy sparse matrix to PyTorch tensor
modified_adj_torch = torch.tensor(modified_adj.toarray(), dtype=torch.float)

# Now you can safely call .cpu() if needed (useful if working with CUDA tensors)
modified_adj_torch = modified_adj_torch.cpu()

modified_adj = csr_matrix(modified_adj_torch)

number of pertubations: 253


In [ ]:
atk_model = SimPGCN(nnodes=features.shape[0], nfeat=features.shape[1],
    nhid=16, nclass=labels.max()+1, device=device).to(device)
atk_model.fit(features, modified_adj, labels, idx_train, idx_val, patience=100, verbose=True)

atk_acc = atk_model.test(idx_test)
print(atk_acc)

=== training gcn model ===
loading saved_knn/knn_graph_(2485, 1433).npz...
loading saved_knn/cosine_sims_(2485, 1433).npy
loading saved_knn/attrsim_sampled_idx_(2485, 1433).npy
number of sampled: 21972
Epoch 0, training loss: 2.1042580604553223
Epoch 10, training loss: 1.226519227027893
Epoch 20, training loss: 0.5335997343063354
Epoch 30, training loss: 0.2053990364074707
Epoch 40, training loss: 0.10306744277477264
Epoch 50, training loss: 0.054293014109134674
Epoch 60, training loss: 0.03120998851954937
Epoch 70, training loss: 0.04105347767472267
Epoch 80, training loss: 0.031713783740997314
Epoch 90, training loss: 0.02100321464240551
Epoch 100, training loss: 0.026865530759096146
Epoch 110, training loss: 0.01693514734506607
Epoch 120, training loss: 0.0271962471306324
=== early stopping at 125, loss_val = 0.5914008021354675 ===
Test set results: loss= 0.6388 accuracy= 0.8048
0.8048289738430584


In [ ]:
print((atk_acc - test_accuracy)*100)

-2.062374245472831


## GCN

In [ ]:
# Setup Surrogate model
surrogate = GCN(nfeat=features.shape[1], nclass=labels.max().item()+1,
                nhid=64, dropout=0, with_relu=True, with_bias=True, device=device).to(device)
surrogate.fit(features, adj, labels, idx_train, idx_val, patience=100, verbose=True)

=== training gcn model ===
Epoch 0, training loss: 2.0254876613616943
Epoch 10, training loss: 0.16422221064567566
Epoch 20, training loss: 0.019253380596637726
Epoch 30, training loss: 0.0077318125404417515
Epoch 40, training loss: 0.00915240217000246
Epoch 50, training loss: 0.013718227855861187
Epoch 60, training loss: 0.016572128981351852
Epoch 70, training loss: 0.015872934833168983
Epoch 80, training loss: 0.014424710534512997
Epoch 90, training loss: 0.013599131256341934
Epoch 100, training loss: 0.013069466687738895
Epoch 110, training loss: 0.012576715089380741
=== early stopping at 113, loss_val = 0.4683011770248413 ===


In [ ]:
preds=surrogate.predict(features,adj)
_, predicted_classes = torch.max(preds, 1)  # Returns values, indices; we need indices

# Step 3: Subset for test nodes
test_preds = predicted_classes[idx_test].cpu().numpy()
test_labels = labels[idx_test]

# Step 4: Calculate accuracy
correct = (test_preds == test_labels).sum().item()  # Count correct predictions
total = idx_test.shape[0]  # Total number of test nodes
test_accuracy = correct / total

print(f'Test Accuracy: {test_accuracy}')

Test Accuracy: 0.8395372233400402


In [ ]:
# Setup Attack Model
model = Random(surrogate, nnodes=adj.shape[0], attack_structure=True, attack_features=False, device=device).to(device)
model.attack(adj, n_perturbations=budget)
modified_adj = model.modified_adj
# Convert SciPy sparse matrix to PyTorch tensor
modified_adj_torch = torch.tensor(modified_adj.toarray(), dtype=torch.float)

# Now you can safely call .cpu() if needed (useful if working with CUDA tensors)
modified_adj_torch = modified_adj_torch.cpu()

modified_adj = csr_matrix(modified_adj_torch)

In [ ]:
atk_model = GCN(nfeat=features.shape[1], nclass=labels.max().item()+1,
                nhid=64, dropout=0, with_relu=True, with_bias=True, device=device).to(device)
atk_model.fit(features, modified_adj, labels, idx_train, idx_val, patience=100, verbose=True)

atk_acc = atk_model.test(idx_test)
print(atk_acc)

=== training gcn model ===
Epoch 0, training loss: 1.965752363204956
Epoch 10, training loss: 0.14442315697669983
Epoch 20, training loss: 0.01634359359741211
Epoch 30, training loss: 0.007796036545187235
Epoch 40, training loss: 0.010289414785802364
Epoch 50, training loss: 0.015795765444636345
Epoch 60, training loss: 0.01812082901597023
Epoch 70, training loss: 0.016661083325743675
Epoch 80, training loss: 0.015030885115265846
Epoch 90, training loss: 0.014180232770740986
Epoch 100, training loss: 0.013568446040153503
Epoch 110, training loss: 0.012998247519135475
=== early stopping at 112, loss_val = 0.465155690908432 ===
Test set results: loss= 0.5287 accuracy= 0.8290
0.8289738430583502


In [ ]:
print((atk_acc - benchmark_clean)*100)

-1.8611670020120652


## GCNSVD

In [ ]:
surrogate2 = GCNSVD(nfeat=features.shape[1],
          nhid=64,
          nclass=labels.max().item() + 1,
          dropout=0.5, device=device).to(device)
surrogate2.fit(features, adj, labels, idx_train, idx_val, k=50)

=== GCN-SVD: rank=50 ===
rank_after = 50
=== training gcn model ===
Epoch 0, training loss: 2.0072333812713623
Epoch 10, training loss: 0.4592343866825104
Epoch 20, training loss: 0.2288334220647812
Epoch 30, training loss: 0.1471269577741623
Epoch 40, training loss: 0.1256720870733261
Epoch 50, training loss: 0.10384183377027512
Epoch 60, training loss: 0.10801202058792114
Epoch 70, training loss: 0.10334079712629318
Epoch 80, training loss: 0.09238924086093903
Epoch 90, training loss: 0.08405647426843643
Epoch 100, training loss: 0.07731684297323227
Epoch 110, training loss: 0.07937982678413391
Epoch 120, training loss: 0.07834400981664658
Epoch 130, training loss: 0.07717201113700867
Epoch 140, training loss: 0.07578078657388687
Epoch 150, training loss: 0.07002373784780502
Epoch 160, training loss: 0.06499943137168884
Epoch 170, training loss: 0.06508328020572662
Epoch 180, training loss: 0.05942505970597267
Epoch 190, training loss: 0.05865178257226944
=== picking the best model a

In [ ]:
preds=surrogate2.predict(features,adj)
_, predicted_classes = torch.max(preds, 1)  # Returns values, indices; we need indices

# Step 3: Subset for test nodes
test_preds = predicted_classes[idx_test].cpu().numpy()
test_labels = labels[idx_test]

# Step 4: Calculate accuracy
correct = (test_preds == test_labels).sum().item()  # Count correct predictions
total = idx_test.shape[0]  # Total number of test nodes
test_accuracy = correct / total

print(f'Test Accuracy: {test_accuracy}')

=== GCN-SVD: rank=50 ===
rank_after = 50
Test Accuracy: 0.7776659959758552


In [ ]:
# Setup Attack Model
model = Random(surrogate2, nnodes=adj.shape[0], attack_structure=True, attack_features=False, device=device).to(device)
model.attack(adj, n_perturbations=budget)
modified_adj = model.modified_adj
# Convert SciPy sparse matrix to PyTorch tensor
modified_adj_torch = torch.tensor(modified_adj.toarray(), dtype=torch.float)

# Now you can safely call .cpu() if needed (useful if working with CUDA tensors)
modified_adj_torch = modified_adj_torch.cpu()

modified_adj = csr_matrix(modified_adj_torch)

In [ ]:
atk_model = GCNSVD(nfeat=features.shape[1],
          nhid=64,
          nclass=labels.max().item() + 1,
          dropout=0.5, device=device).to(device)
atk_model.fit(features, modified_adj, labels, idx_train, idx_val, patience=100, verbose=True)

atk_acc = atk_model.test(idx_test)
print(atk_acc)

=== GCN-SVD: rank=50 ===
rank_after = 50
=== training gcn model ===
Epoch 0, training loss: 1.9951658248901367
Epoch 10, training loss: 0.4771680533885956
Epoch 20, training loss: 0.24277465045452118
Epoch 30, training loss: 0.1674921065568924
Epoch 40, training loss: 0.14359399676322937
Epoch 50, training loss: 0.10838336497545242
Epoch 60, training loss: 0.11508247256278992
Epoch 70, training loss: 0.10182808339595795
Epoch 80, training loss: 0.10541876405477524
Epoch 90, training loss: 0.09216145426034927
Epoch 100, training loss: 0.08674002438783646
Epoch 110, training loss: 0.08656434714794159
Epoch 120, training loss: 0.0795908272266388
Epoch 130, training loss: 0.07393693178892136
Epoch 140, training loss: 0.07294539362192154
Epoch 150, training loss: 0.07418721914291382
Epoch 160, training loss: 0.06426604837179184
Epoch 170, training loss: 0.06187780946493149
Epoch 180, training loss: 0.06268531829118729
Epoch 190, training loss: 0.06909627467393875
=== picking the best model 

In [ ]:
print((atk_acc - benchmark_clean)*100)

-8.702213279678062


### GCN Jaccard

In [ ]:
surrogate1 = GCNJaccard(nfeat=features.shape[1],
          nhid=64,
          nclass=labels.max().item() + 1,
          dropout=0.5, device=device).to(device)
surrogate1.fit(features, adj, labels, idx_train, idx_val, threshold=0.03)

removed 1015 edges in the original graph
=== training gcn model ===
Epoch 0, training loss: 1.9383213520050049
Epoch 10, training loss: 0.17206737399101257
Epoch 20, training loss: 0.04792378470301628
Epoch 30, training loss: 0.01681855134665966
Epoch 40, training loss: 0.013223609887063503
Epoch 50, training loss: 0.015077665448188782
Epoch 60, training loss: 0.020669620484113693
Epoch 70, training loss: 0.02149527333676815
Epoch 80, training loss: 0.02091405913233757
Epoch 90, training loss: 0.01609087735414505
Epoch 100, training loss: 0.01990603655576706
Epoch 110, training loss: 0.014299966394901276
Epoch 120, training loss: 0.014565463177859783
Epoch 130, training loss: 0.02086189202964306
Epoch 140, training loss: 0.013033526949584484
Epoch 150, training loss: 0.014002456329762936
Epoch 160, training loss: 0.013720656745135784
Epoch 170, training loss: 0.0118795745074749
Epoch 180, training loss: 0.013923942111432552
Epoch 190, training loss: 0.01620454154908657
=== picking the 

In [ ]:
preds=surrogate1.predict(features,adj)
_, predicted_classes = torch.max(preds, 1)  # Returns values, indices; we need indices

# Step 3: Subset for test nodes
test_preds = predicted_classes[idx_test].cpu().numpy()
test_labels = labels[idx_test]

# Step 4: Calculate accuracy
correct = (test_preds == test_labels).sum().item()  # Count correct predictions
total = idx_test.shape[0]  # Total number of test nodes
test_accuracy = correct / total

print(f'Test Accuracy: {test_accuracy}')

removed 1015 edges in the original graph
Test Accuracy: 0.8003018108651911


In [ ]:
# Setup Attack Model
model = Random(surrogate1, nnodes=adj.shape[0], attack_structure=True, attack_features=False, device=device).to(device)
model.attack(adj, n_perturbations=budget)

modified_adj = model.modified_adj
# Convert SciPy sparse matrix to PyTorch tensor
modified_adj_torch = torch.tensor(modified_adj.toarray(), dtype=torch.float)

# Now you can safely call .cpu() if needed (useful if working with CUDA tensors)
modified_adj_torch = modified_adj_torch.cpu()

modified_adj = csr_matrix(modified_adj_torch)

In [ ]:
atk_model = GCNJaccard(nfeat=features.shape[1],
          nhid=64,
          nclass=labels.max().item() + 1,
          dropout=0.5, device=device).to(device)
atk_model.fit(features, modified_adj, labels, idx_train, idx_val, patience=100, verbose=True)

atk_acc = atk_model.test(idx_test)
print(atk_acc)

removed 647 edges in the original graph
=== training gcn model ===
Epoch 0, training loss: 2.0030083656311035
Epoch 10, training loss: 0.22502483427524567
Epoch 20, training loss: 0.04332933574914932
Epoch 30, training loss: 0.020152686163783073
Epoch 40, training loss: 0.025442497804760933
Epoch 50, training loss: 0.024266783148050308
Epoch 60, training loss: 0.01981409080326557
Epoch 70, training loss: 0.02382407709956169
Epoch 80, training loss: 0.020334556698799133
Epoch 90, training loss: 0.018671700730919838
Epoch 100, training loss: 0.01565883681178093
Epoch 110, training loss: 0.02035396173596382
Epoch 120, training loss: 0.01908489689230919
Epoch 130, training loss: 0.017074301838874817
Epoch 140, training loss: 0.018911724910140038
Epoch 150, training loss: 0.01562618277966976
Epoch 160, training loss: 0.0171397365629673
Epoch 170, training loss: 0.017664574086666107
Epoch 180, training loss: 0.015551199205219746
Epoch 190, training loss: 0.013927684165537357
=== picking the 

In [ ]:
benchmark_clean

0.8475855130784709

In [ ]:
print((atk_acc - benchmark_clean)*100)

-2.766599597585506


In [ ]:
# Setup Surrogate model

features = normalize_feature(features)
surrogate = GCN(nfeat=features.shape[1], nclass=labels.max().item()+1,
                nhid=64, dropout=0, with_relu=True, with_bias=True, device=device).to(device)
surrogate.fit(features, adj, labels, idx_train, idx_val, patience=100, verbose=True)


preds=surrogate.predict(features,adj)
_, predicted_classes = torch.max(preds, 1)  # Returns values, indices; we need indices

# Step 3: Subset for test nodes
test_preds = predicted_classes[idx_test].cpu().numpy()
test_labels = labels[idx_test]

# Step 4: Calculate accuracy
correct = (test_preds == test_labels).sum().item()  # Count correct predictions
total = idx_test.shape[0]  # Total number of test nodes
test_accuracy = correct / total

print(f'Test Accuracy: {test_accuracy}')




NameError: name 'sp' is not defined

In [ ]:
# Setup Attack Model
model1 = PGDAttack(surrogate, nnodes=adj.shape[0], attack_structure=True, attack_features=False, device=device).to(device)
# Attack
model1.attack(features, adj, labels, idx_train, n_perturbations=budget)
modified_adj = model1.modified_adj # modified_adj is a torch.tensor


  0%|          | 0/200 [00:00<?, ?it/s]


NotImplementedError: Could not run 'aten::fill_.Scalar' with arguments from the 'SparseCUDA' backend. This could be because the operator doesn't exist for this backend, or was omitted during the selective/custom build process (if using custom build). If you are a Facebook employee using PyTorch on mobile, please visit https://fburl.com/ptmfixes for possible resolutions. 'aten::fill_.Scalar' is only available for these backends: [CPU, CUDA, Meta, QuantizedCPU, QuantizedCUDA, SparseCsrCPU, SparseCsrCUDA, NestedTensorCPU, NestedTensorCUDA, BackendSelect, Python, FuncTorchDynamicLayerBackMode, Functionalize, Named, Conjugate, Negative, ZeroTensor, ADInplaceOrView, AutogradOther, AutogradCPU, AutogradCUDA, AutogradHIP, AutogradXLA, AutogradMPS, AutogradIPU, AutogradXPU, AutogradHPU, AutogradVE, AutogradLazy, AutogradMTIA, AutogradPrivateUse1, AutogradPrivateUse2, AutogradPrivateUse3, AutogradMeta, AutogradNestedTensor, Tracer, AutocastCPU, AutocastCUDA, FuncTorchBatched, BatchedNestedTensor